# Securing Data in Unity Catalog

In this demo you will learn how to hide sensitive data using 3 different approaches:
* Views 
* Dynamic Views
* Row Filter and Column Masks on Tables (introduced in 2024)

Further, you will also learn data governance features of Unity Catalog
* Introduction to Catalog Explorer 
* Enable data access to users using inherited and explicit privileges 
* Tagging + AI generated Documentation
* Use Lineage and Insight features in Unity Catalog to understand data flow and access patterns.

------------------------------------------------------------------------------

## Row Filters and Column Masks (Introduced in 2024) 

This newly introduced feature enables data owners to mask columns and hide rows in similar ways to Dynamic Views - without having to create another data object.

- **Row filters** allow you to apply a filter to a table so that queries return only rows that meet the filter criteria. You implement a row filter as a SQL user-defined function (UDF). Python and Scala UDFs are also supported, but only when they are wrapped in a SQL UDF.

- **Column masks** let you apply a masking function to a table column. The masking function gets evaluated at query runtime, substituting each reference of the target column with the results of the masking function. For most use cases, column masks determine whether to return the original column value or redact it based on the identity of the invoking user. Column masks are expressions written as SQL UDFs or as Python or Scala UDFs that are wrapped in a SQL UDF.

In [0]:
spark.sql("Create catalog if not exists dbx_catalog")
spark.sql("Create schema if not exists dbx_catalog.dbx_schema")

DataFrame[]

In [0]:
%sql
use catalog dbx_catalog;
use schema dbx_schema;

In [0]:
spark.sql("grant use catalog  on catalog  dbx_catalog to Fgroup")
spark.sql("grant use schema  on schema dbx_catalog.dbx_schema to Fgroup")

DataFrame[]

In [0]:
%sql
create table if not exists Checking_PII_tables
( ID INT,
  NAME STRING,
  CREDIT_CARD  STRING)

In [0]:
%sql
INSERT INTO Checking_PII_tables values 
(1,'santanu1','1231-1231-1231-1231'),
(1,'santanu2','2231-2231-2231-2231'),
(1,'santanu3','3231-3231-3231-3231')

![](path)

In [0]:
#sql("revoke select on table  dbx_catalog.dbx_schema.Checking_PII_tables from  Fgroup")

In [0]:
sql("grant select on table  dbx_catalog.dbx_schema.Checking_PII_tables to Fgroup")

In [0]:
%sql
CREATE OR REPLACE FUNCTION  card_mask(CREDIT_CARD STRING)
  RETURN CASE WHEN is_account_group_member('Fgroup') THEN  '****-****-****-****' ELSE CREDIT_CARD END;

In [0]:
%sql
select * from Checking_PII_tables

In [0]:
%sql
alter table Checking_PII_tables ALTER COLUMN CREDIT_CARD SET MASK dbx_catalog.dbx_schema.card_mask

In [0]:
%sql
SELECT *
    FROM information_schema.column_masks
    WHERE table_catalog = 'dbx_catalog'
    AND table_schema    = 'dbx_schema'
    AND table_name      = 'Checking_PII_tables'

table_catalog,table_schema,table_name,column_name,mask_name,using_columns


In [0]:
%sql
create or replace table  table_for_dept_row_filter
( ID INT,
  NAME STRING,
  CREDIT_CARD  STRING,
  dept STRING);
  -----------------------------------------------------------------------

  INSERT INTO table_for_dept_row_filter values 
(1,'santanu1','1231-1231-1231-1231',"HR"),
(2,'santanu2','2231-2231-2231-2231',"FINANCE"),
(3,'santanu3','3231-3231-3231-3231',"FINANCE");

select * from table_for_dept_row_filter;

ID,NAME,CREDIT_CARD,dept
1,santanu1,1231-1231-1231-1231,HR
2,santanu2,2231-2231-2231-2231,FINANCE
3,santanu3,3231-3231-3231-3231,FINANCE


In [0]:
%sql
select is_account_group_member('Fgroup'),current_user(),is_member('users');

is_account_group_member('Fgroup'),current_user(),is_member('users')
false,sandutta2020@gmail.com,true


In [0]:
%sql
DROP FUNCTION IF EXISTS dept_row_filter;
-------IF(Condition,what happens when condition returns true, what happends when conditions return false)
CREATE OR REPLACE FUNCTION dept_row_filter(dept STRING)
RETURNS BOOLEAN
RETURN IF(is_account_group_member('Fgroup'), dept ='FINANCE',false);

In [0]:
%sql
ALTER TABLE table_for_dept_row_filter 
SET ROW FILTER dept_row_filter ON (dept);

In [0]:
%sql
select * from table_for_dept_row_filter

ID,NAME,CREDIT_CARD,dept
2,santanu2,2231-2231-2231-2231,FINANCE
3,santanu3,3231-3231-3231-3231,FINANCE


In [0]:
%sql
SELECT *
    FROM information_schema.row_filters
    WHERE table_catalog = 'dbx_catalog'
    AND table_schema    = 'dbx_schema'
    AND table_name      = 'table_for_dept_row_filter'

table_catalog,table_schema,table_name,filter_name,target_columns
dbx_catalog,dbx_schema,table_for_dept_row_filter,dbx_catalog.dbx_schema.dept_row_filter,dept


### D2. Dynamic Views

We have seen that Unity Catalog's treatment of views provides the ability for views to protect access to tables; users can be granted access to views that manipulate, transform, or obscure data from a source table, without needing to provide direct access to the source table.

Dynamic views provide the ability to do fine-grained access control of columns and rows within a table, conditional on the principal running the query. Dynamic views are an extension to standard views that allow us to do things like:
* partially obscure column values or redact them entirely
* omit rows based on specific criteria

Access control with dynamic views is achieved through the use of functions within the definition of the view. These functions include:
* `current_user()`: returns the email address of the user querying the view
* `is_account_group_member()`: returns TRUE if the user querying the view is a member of the specified group
* `is_member()`: returns TRUE if the user querying the view is a member of the specified workspace-local group

**NOTE:** Databricks generally advises against using the `is_member()` function in production, since it references workspace-local groups and hence introduces a workspace dependency into a metastore that potentially spans multiple workspaces.

In [0]:
%sql
CREATE OR REPLACE VIEW customers_gold_dynamic_view AS
SELECT 
  CASE WHEN        -- Redact customer_id column if user is not a supervisor
    is_account_group_member('Fgroup') THEN CREDIT_CARD 
    ELSE '****-****-****-****'
  END AS Credit_card,
  ID, 
  NAME, 
  dept
FROM table_for_dept_row_filter
WHERE
  CASE WHEN          -- Redact rows where loyalty_segment 3 or above if the user is not a supervisor
    is_account_group_member('Fgroup') THEN dept ='FINANCE'  -- When true, return all rows
    ELSE  false                          -- When false, return rows less than 3
  END
ORDER BY ID;

In [0]:
%sql
select * from customers_gold_dynamic_view ---There should be no records

Credit_card,ID,NAME,dept
2231-2231-2231-2231,2,santanu2,FINANCE
3231-3231-3231-3231,3,santanu3,FINANCE


##  Tagging 
Tags are attributes with keys and optional values that can be applied to securable objects in Unity Catalog to organize and categorize them.

- Supported objects for tagging include catalogs, schemas, tables, columns, volumes, views, registered models, and model versions.

- Tags simplify search and discovery of tables and views using workspace search functionality.

- You can assign up to 20 tags per object, with key length up to 255 characters and value length up to 1000 characters.

- Tags can be added and managed through Catalog Explorer UI or SQL commands (for Databricks Runtime 13.3+).

- Tags can be used for data classification, security, lifecycle management, compliance, and project management.

In [0]:
%sql
ALTER TABLE table_for_dept_row_filter 
SET TAGS (
  'quality'='gold',
  'domain'='customer'
  );


-- COLUMN TAGS
ALTER TABLE table_for_dept_row_filter 
  ALTER COLUMN CREDIT_CARD SET TAGS ("compliance" = "GDPR");

In [0]:
%sql
SELECT * 
FROM INFORMATION_SCHEMA.TABLE_TAGS
WHERE TABLE_NAME = 'table_for_dept_row_filter'

catalog_name,schema_name,table_name,tag_name,tag_value
dbx_catalog,dbx_schema,table_for_dept_row_filter,domain,customer
dbx_catalog,dbx_schema,table_for_dept_row_filter,quality,gold


Alternatively, run the query below leveraging the `INFORMATION_SCHEMA.TABLE_TAGS` filtering with the `customers_silver` table.
Be aware of the tables below to retrieve tags from the different objects:

- `INFORMATION_SCHEMA.CATALOG_TAGS`
- `INFORMATION_SCHEMA.SCHEMA_TAGS`
- `INFORMATION_SCHEMA.TABLE_TAGS`
- `INFORMATION_SCHEMA.COLUMN_TAGS`
- `INFORMATION_SCHEMA.VOLUME_TAGS`


## Lineage

1. Data lineage is a key pillar of any data governance solution. In the **Lineage** tab, we can identify elements that are related to the selected object:
* With **Upstream** selected, we see objects that gave rise to this object, or that this object uses. This is useful for tracing the source of your data.
* With **Downstream** selected, we see objects that are using this object. This is useful for performing impact analyses.
* The lineage graph provides a visualization of the lineage relationships.

You can access the lineage of a table in Catalog Explorer by selecting your table, in this case **customers_silver**, in the _"lineage"_ tab there is a button _"see lineage graph"_ to display the results shown below. 

**Example**

**Note:** Take into consideration some information such as catalog won't match your view.

## AI generated Documentation

AI-generated documentation for Unity Catalog enables automatically generate descriptions for tables and columns. The feature uses a custom-built large language model (LLM) to generate metadata based on table schemas and column names.

- Available for catalogs, schemas, tables, columns, functions, models, and volumes.
- Saves time and reduces manual effort in documenting data assets.
- Improves search functionality within Databricks workspaces.
- Users need appropriate permissions (object owner or MODIFY privilege) to view, edit, and save AI-generated comments.